[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-fairness.ipynb)

# ML Fairness & Bias Auditing

*AIBits Academy · Machine Learning End To End · Responsible ML · New*

A model can be 95% accurate and still systematically disadvantage a protected group. This closing chapter covers measuring that risk directly, rather than assuming accuracy alone is enough.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## How Bias Enters a Model Without Anyone Intending It

| Source | Mechanism | Example |
|---|---|---|
| Historical bias | Training labels reflect past discriminatory human decisions | Historical loan approvals under-represent a community due to past lending bias — the model learns to repeat it |
| Representation bias | Training data under-samples a subgroup | A hiring model trained mostly on past hires from one demographic learns weaker signal for others |
| Proxy variables | A feature correlates strongly with a protected attribute even when that attribute is excluded | Pincode strongly correlates with religion/caste in parts of India — excluding "religion" directly doesn't remove the signal if pincode remains |
| Measurement bias | The label itself is a biased proxy for what you actually want to measure | "Arrests" as a proxy for "committed crime" bakes in policing-pattern bias, not actual crime rate |

## Fairness Metrics — There Is No Single "Fair"

A foundational, uncomfortable result in fairness research: several intuitively reasonable fairness definitions are **mathematically incompatible** with each other whenever the base rates genuinely differ between groups — you generally cannot satisfy all of them simultaneously. This forces an explicit choice, not a default.

$$\begin{gathered}\text{Demographic Parity: } P(\hat{y}=1\mid A=a) = P(\hat{y}=1\mid A=b) \quad \text{(equal approval RATE across groups)} \\[6pt] \text{Equal Opportunity: } P(\hat{y}=1\mid y=1,A=a) = P(\hat{y}=1\mid y=1,A=b) \quad \text{(equal TRUE POSITIVE RATE across groups)} \\[6pt] \text{Predictive Parity: } P(y=1\mid \hat{y}=1,A=a) = P(y=1\mid \hat{y}=1,A=b) \quad \text{(equal PRECISION across groups)}\end{gathered}$$

Demographic Parity asks "do both groups get approved at the same rate?" — appropriate when equal access/opportunity outcome is the goal. Equal Opportunity asks "among genuinely qualified applicants, are both groups caught at the same rate?" — appropriate when you want to avoid disadvantaging qualified members of any group. These can conflict directly whenever the true qualification rate differs between groups for reasons unrelated to the model.

## Code — Auditing a Credit Model Across Groups

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

np.random.seed(15)
n = 2000
group = np.random.choice(['GroupA','GroupB'], n, p=[0.6,0.4])
# Simulate a HISTORICAL bias: GroupB's true qualification is equal, but observed features are noisier for them
income = np.random.normal(45,15,n)
true_qualified = (income > 40).astype(int)
noisy_cibil = np.random.normal(700,60,n) - np.where(group=='GroupB', 25, 0)  # historically noisier scoring

X = np.column_stack([income, noisy_cibil])
X_tr,X_te,y_tr,y_te,g_tr,g_te = train_test_split(X, true_qualified, group, test_size=0.3, random_state=42)
clf = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
preds = clf.predict(X_te)

results = pd.DataFrame({'group':g_te, 'y_true':y_te, 'y_pred':preds})
for grp in ['GroupA','GroupB']:
    sub = results[results.group==grp]
    approval_rate = sub.y_pred.mean()
    # Equal Opportunity: true positive rate among genuinely qualified applicants
    qualified = sub[sub.y_true==1]
    tpr = qualified.y_pred.mean() if len(qualified) else float('nan')
    print(f"{grp}:  approval_rate={approval_rate:.3f}   TPR (equal opportunity)={tpr:.3f}")

Both groups have identical *true* qualification rates by construction in this simulation — the gap (89.1% vs 76.2% TPR) is purely an artefact of the noisier historical scoring for Group B baked into the training data. This is exactly how historical bias silently propagates into a model that never explicitly used group membership as a feature at all.

## The Proxy Variable Trap

> **⚠ Removing a Protected Attribute Is Not Enough**
>
> "Fairness through unawareness" — simply excluding religion/caste/gender as a direct model input — is widely known to be insufficient. If other included features (pincode, surname, school attended, even certain spending patterns) correlate strongly with a protected attribute, the model can reconstruct and rely on that signal indirectly, achieving materially the same disparate outcome as if the attribute had been included directly.

In [ ]:
# Detecting a potential proxy: does a "neutral" feature strongly predict group membership?
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Can pincode alone predict group membership well above chance?
pincode_proxy_model = LogisticRegression().fit(X_tr[:,[1]], (g_tr=='GroupB').astype(int))
proxy_auc = roc_auc_score((g_te=='GroupB').astype(int),
                          pincode_proxy_model.predict_proba(X_te[:,[1]])[:,1])
print(f"AUC of 'noisy_cibil' predicting group membership: {proxy_auc:.3f}")
# AUC well above 0.5 → this feature is acting as a partial proxy for group, worth investigating

## Mitigation Strategies

| Stage | Technique |
|---|---|
| Pre-processing | Reweight or resample training data so groups are represented proportionally to their true (not historically-observed) qualification rates |
| In-processing | Add a fairness penalty term to the training objective (fairness-constrained optimisation) |
| Post-processing | Apply group-specific decision thresholds so a chosen fairness metric (e.g., Equal Opportunity) is satisfied post-hoc, without retraining |
| Ongoing | Continuous monitoring — fairness metrics can drift as the underlying population shifts, exactly like accuracy can drift |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Selection rate per group

Given predictions and a sensitive attribute, store in `rates` a dict of the **approval (selection) rate** of each group, and `dp_diff` = the absolute difference between the two groups (demographic-parity difference).

In [ ]:
import numpy as np
group = np.array(["A"] * 6 + ["B"] * 4)
pred = np.array([1, 1, 1, 0, 1, 0, 1, 0, 0, 0])
rates = {}
dp_diff = None   # TODO


In [ ]:
try:
    check("group rates", rates == {"A": 4 / 6, "B": 1 / 4})
    check("difference", abs(dp_diff - (4 / 6 - 1 / 4)) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
group = np.array(["A"] * 6 + ["B"] * 4)
pred = np.array([1, 1, 1, 0, 1, 0, 1, 0, 0, 0])
rates = {g: float(pred[group == g].mean()) for g in ["A", "B"]}
dp_diff = abs(rates["A"] - rates["B"])

```

</details>

### Exercise 2 · Medium · Equal opportunity

Among people who **truly qualify** (`y_true == 1`), what share does the model approve in each group? Store the true-positive rates in `tpr` (dict) and the gap in `eo_gap`.

In [ ]:
y_true = np.array([1, 1, 1, 0, 1, 1, 1, 0, 1, 0])
tpr = {}
eo_gap = None   # TODO (reuse group, pred)


In [ ]:
try:
    check("TPR for A", abs(tpr["A"] - 0.8) < 1e-12)
    check("TPR for B", abs(tpr["B"] - 0.5) < 1e-12)
    check("gap", abs(eo_gap - 0.3) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
y_true = np.array([1, 1, 1, 0, 1, 1, 1, 0, 1, 0])
tpr = {g: float(pred[(group == g) & (y_true == 1)].mean()) for g in ["A", "B"]}
eo_gap = abs(tpr["A"] - tpr["B"])

```

</details>

### Exercise 3 · Stretch · The four-fifths rule

Compute the **disparate-impact ratio** = (lowest group selection rate) / (highest group selection rate) from `rates`. Store it in `di` and set `passes` to whether it is at least 0.8 (the common regulatory rule of thumb).

In [ ]:
di = passes = None   # TODO (reuse rates)


In [ ]:
try:
    check("ratio", abs(di - (0.25 / (4 / 6))) < 1e-12)
    check("fails the 4/5 rule", passes is False)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
di = min(rates.values()) / max(rates.values())
passes = bool(di >= 0.8)

```

Different fairness metrics can disagree — which one applies is a policy decision, not a modelling one.

</details>

---
*Back to the course: **Machine Learning End To End → ML Fairness & Bias Auditing**.*